In [2]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# ============================================================
# GOOGLE SHEET CONFIGURATION
# ============================================================

SHEET_ID = "1-fkXH6w6ehXRa6AXP-qProhjAeTWVj8BmMJHwhZa5Ho"

# CORRECT GIDs
RA_GID = "0"
MASTER_GID = "1836034777"

# ============================================================
# LIVE GOOGLE SHEET CONNECTOR
# ============================================================

def get_live_google_data():

    master_url = (
        f"https://docs.google.com/spreadsheets/d/"
        f"{SHEET_ID}/export?format=csv&gid={MASTER_GID}"
    )

    ra_url = (
        f"https://docs.google.com/spreadsheets/d/"
        f"{SHEET_ID}/export?format=csv&gid={RA_GID}"
    )

    master_df = pd.read_csv(master_url)
    ra_df = pd.read_csv(ra_url)

    master_df.columns = master_df.columns.astype(str).str.strip()
    ra_df.columns = ra_df.columns.astype(str).str.strip()

    return master_df, ra_df

# ============================================================
# COLUMN FINDER
# ============================================================

def find_column(df, keyword):

    for col in df.columns:

        if keyword.lower() in str(col).lower():

            return col

    return None

# ============================================================
# ELECTION GPT ENGINE
# ============================================================

def gpt_query_engine(constituency, question):

    try:

        master_df, ra_df = get_live_google_data()

        constituency_col = find_column(
            ra_df,
            "constituency"
        )

        candidate_col = find_column(
            ra_df,
            "candidate"
        )

        votes_col = find_column(
            ra_df,
            "vote"
        )

        round_col = find_column(
            ra_df,
            "round"
        )

        alert_col = find_column(
            ra_df,
            "alert"
        )

        ra_df[constituency_col] = (
            ra_df[constituency_col]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        master_df["Constituency"] = (
            master_df["Constituency"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        con = constituency.lower()

        c_data = ra_df[
            ra_df[constituency_col] == con
        ]

        if c_data.empty:

            return (
                f"❌ No data found for "
                f"{constituency}"
            )

        q = question.lower()

        # =====================================================
        # LEADING CANDIDATE
        # =====================================================

        if (
            "leading" in q
            or "winner" in q
            or "winning" in q
        ):

            leader_table = (
                c_data.groupby(candidate_col)[votes_col]
                .sum()
                .sort_values(
                    ascending=False
                )
            )

            leader = leader_table.index[0]

            votes = leader_table.iloc[0]

            return (
                f"🏆 LEADING CANDIDATE\n\n"
                f"Constituency : {constituency}\n"
                f"Leader       : {leader}\n"
                f"Votes        : {votes}"
            )

        # =====================================================
        # STATUS / MISSING ROUND
        # =====================================================

        elif (
            "status" in q
            or "missing" in q
        ):

            tracker = master_df[
                master_df["Constituency"]
                == con
            ]

            if len(tracker) == 0:

                return (
                    "❌ Constituency not found "
                    "in MASTER_TRACKER"
                )

            latest_round = (
                tracker.iloc[0]
                ["Latest Round"]
            )

            leader = (
                tracker.iloc[0]
                ["Leading Candidate"]
            )

            missing = (
                tracker.iloc[0]
                ["Missing round"]
            )

            return (
                f"📊 CONSTITUENCY STATUS\n\n"
                f"Constituency : {constituency}\n"
                f"Latest Round : {latest_round}\n"
                f"Leader       : {leader}\n"
                f"Missing      : {missing}"
            )

        # =====================================================
        # ALERT REPORT
        # =====================================================

        elif (
            "alert" in q
            or "delay" in q
            or "overtime" in q
        ):

            if alert_col is None:

                return (
                    "❌ Alert Status "
                    "column missing."
                )

            overtime = c_data[
                c_data[alert_col]
                .astype(str)
                .str.contains(
                    "overtime",
                    case=False,
                    na=False
                )
            ]

            return (
                f"🚨 ALERT REPORT\n\n"
                f"Constituency : {constituency}\n"
                f"Delayed Rows : "
                f"{len(overtime)}"
            )

        # =====================================================
        # VOTE SUMMARY
        # =====================================================

        else:

            summary = (
                c_data.groupby(
                    candidate_col
                )[votes_col]
                .sum()
                .sort_values(
                    ascending=False
                )
            )

            return (
                f"📋 LIVE VOTE TOTALS\n\n"
                f"{summary.to_string()}"
            )

    except Exception as e:

        return (
            f"❌ Error : {str(e)}"
        )

# ============================================================
# LOAD DROPDOWN OPTIONS
# ============================================================

try:

    _, ra_df = get_live_google_data()

    constituency_col = find_column(
        ra_df,
        "constituency"
    )

    constituency_list = sorted(
        ra_df[constituency_col]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

except:

    constituency_list = [
        "Mettuppalayam",
        "Vandavasi",
        "Killiyoor",
        "Sattur",
        "Omalur",
        "Papanasam"
    ]

# ============================================================
# UI
# ============================================================

constituency_dropdown = widgets.Dropdown(
    options=constituency_list,
    description="Constituency:",
    layout=widgets.Layout(
        width="300px"
    )
)

question_input = widgets.Text(
    placeholder=
    "Who is leading?",
    description="Question:",
    layout=widgets.Layout(
        width="500px"
    )
)

btn_ask = widgets.Button(
    description=
    "Ask Election GPT",
    button_style=
    "primary"
)

btn_refresh = widgets.Button(
    description=
    "Refresh Data",
    button_style=
    "success"
)

out = widgets.Output(
    layout={
        "border":
        "1px solid #4a90e2",
        "padding":
        "15px"
    }
)

# ============================================================
# BUTTON EVENTS
# ============================================================

def on_ask(b):

    with out:

        clear_output(
            wait=True
        )

        print(
            f"🤖 Election GPT "
            f"analyzing "
            f"{constituency_dropdown.value}"
            f"...\n"
        )

        result = (
            gpt_query_engine(
                constituency_dropdown.value,
                question_input.value
            )
        )

        print(result)

def on_refresh(b):

    with out:

        clear_output(
            wait=True
        )

        try:

            _, ra_df = (
                get_live_google_data()
            )

            constituency_col = (
                find_column(
                    ra_df,
                    "constituency"
                )
            )

            constituency_dropdown.options = (
                sorted(
                    ra_df[
                        constituency_col
                    ]
                    .dropna()
                    .astype(str)
                    .str.strip()
                    .unique()
                    .tolist()
                )
            )

            print(
                "✅ Google Sheet "
                "Refreshed Successfully"
            )

        except Exception as e:

            print(
                f"❌ Refresh Failed : "
                f"{e}"
            )

btn_ask.on_click(on_ask)
btn_refresh.on_click(on_refresh)

# ============================================================
# DISPLAY
# ============================================================

display(
    widgets.VBox([
        widgets.HTML(
            "<h2>🗳 Election "
            "Operations GPT "
            "Dashboard</h2>"
        ),

        widgets.HBox([
            constituency_dropdown,
            question_input
        ]),

        widgets.HBox([
            btn_ask,
            btn_refresh
        ]),

        out
    ])
)

In [3]:
import pandas as pd

# ============================================================
# GOOGLE SHEET CONFIGURATION
# ============================================================

SHEET_ID = "1-fkXH6w6ehXRa6AXP-qProhjAeTWVj8BmMJHwhZa5Ho"

# CORRECT GIDs
RA_GID = "0"
MASTER_GID = "1836034777"

# ============================================================
# LOAD LIVE GOOGLE SHEET DATA
# ============================================================

def get_live_google_data():

    master_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={MASTER_GID}"

    ra_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={RA_GID}"

    master_df = pd.read_csv(master_url)
    ra_df = pd.read_csv(ra_url)

    master_df.columns = master_df.columns.str.strip()
    ra_df.columns = ra_df.columns.str.strip()

    return master_df, ra_df

# ============================================================
# DEBUG SATTUR
# ============================================================

master_df, ra_df = get_live_google_data()

# Clean constituency names

master_df['Constituency'] = (
    master_df['Constituency']
    .astype(str)
    .str.strip()
    .str.lower()
)

ra_df['Constituency'] = (
    ra_df['Constituency']
    .astype(str)
    .str.strip()
    .str.lower()
)

print("="*60)
print("MASTER TRACKER ENTRY FOR SATTUR")
print("="*60)

sattur_master = master_df[
    master_df['Constituency'] == 'sattur'
]

print(sattur_master)

print("\n")
print("="*60)
print("RA ENTRY PANEL DATA FOR SATTUR")
print("="*60)

sattur_ra = ra_df[
    ra_df['Constituency'] == 'sattur'
]

print(
    sattur_ra[
        ['Constituency','Round']
    ]
)

print("\n")
print("="*60)
print("MISSING ROUND CALCULATION")
print("="*60)

if len(sattur_master) > 0:

    latest = int(
        pd.to_numeric(
            sattur_master.iloc[0]['Latest Round'],
            errors='coerce'
        )
    )

    entered = set(
        pd.to_numeric(
            sattur_ra['Round'],
            errors='coerce'
        )
        .dropna()
        .astype(int)
        .tolist()
    )

    expected = set(
        range(
            1,
            latest + 1
        )
    )

    missing = sorted(
        list(
            expected - entered
        )
    )

    print("Latest Round :", latest)
    print("Entered      :", sorted(entered))
    print("Expected     :", sorted(expected))
    print("Missing      :", missing)

    if len(missing) == 0:

        print("\n✅ STATUS : SYNCHRONIZED")

    else:

        print("\n🚨 STATUS : MISSING ROUNDS FOUND")

else:

    print(
        "❌ SATTUR NOT FOUND IN MASTER TRACKER"
    )

MASTER TRACKER ENTRY FOR SATTUR
  Constituency  Latest Round Leading Candidate      Missing round
3       sattur            11       candidate k  3, 4, 6, 8, 9, 10


RA ENTRY PANEL DATA FOR SATTUR
   Constituency  Round
15       sattur      1
16       sattur      1
17       sattur      1
18       sattur      1
19       sattur      1
65       sattur      7
68       sattur      5
72       sattur     11
73       sattur      2


MISSING ROUND CALCULATION
Latest Round : 11
Entered      : [1, 2, 5, 7, 11]
Expected     : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Missing      : [3, 4, 6, 8, 9, 10]

🚨 STATUS : MISSING ROUNDS FOUND


In [4]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
import time

# =====================================================
# GOOGLE SHEET CONFIGURATION
# =====================================================

SHEET_ID = "1-fkXH6w6ehXRa6AXP-qProhjAeTWVj8BmMJHwhZa5Ho"

# CORRECT GIDs
RA_GID = "0"
MASTER_GID = "1836034777"

# =====================================================
# CONNECT TO LIVE GOOGLE SHEET
# =====================================================

def get_live_google_data():

    master_url = (
        f"https://docs.google.com/spreadsheets/d/"
        f"{SHEET_ID}/export?format=csv&gid={MASTER_GID}"
    )

    ra_url = (
        f"https://docs.google.com/spreadsheets/d/"
        f"{SHEET_ID}/export?format=csv&gid={RA_GID}"
    )

    master_df = pd.read_csv(master_url)
    ra_df = pd.read_csv(ra_url)

    master_df.columns = master_df.columns.str.strip()
    ra_df.columns = ra_df.columns.str.strip()

    return master_df, ra_df

# =====================================================
# ANALYSIS ENGINE
# =====================================================


def generate_dashboard():

    try:

        master_df, ra_df = get_live_google_data()

        master_df['Constituency'] = (
            master_df['Constituency']
            .astype(str)
            .str.strip()
            .str.lower()
        )

        ra_df['Constituency'] = (
            ra_df['Constituency']
            .astype(str)
            .str.strip()
            .str.lower()
        )

        html = f"""
        <h2 style='color:#1565C0'>
        🗳️ Election Operations Dashboard
        </h2>

        <b>Last Refresh:</b> {time.strftime('%H:%M:%S')}
        <br><br>

        <table border='1'
        style='border-collapse:collapse;
               width:100%;
               text-align:center;
               font-family:Arial;'>

        <tr style='background:#1565C0;color:white'>
            <th>Constituency</th>
            <th>Latest Round</th>
            <th>Leader</th>
            <th>Status</th>
            <th>Missing Rounds</th>
        </tr>
        """

        for _, row in master_df.iterrows():

            con = str(row['Constituency']).strip()

            latest = int(
                pd.to_numeric(
                    row['Latest Round'],
                    errors='coerce'
                )
            )

            leader = str(
                row['Leading Candidate']
            )

            temp = ra_df[
                ra_df['Constituency'] == con
            ]

            entered = set(
                pd.to_numeric(
                    temp['Round'],
                    errors='coerce'
                )
                .dropna()
                .astype(int)
            )

            expected = set(
                range(1, latest + 1)
            )

            missing = sorted(
                list(expected - entered)
            )

            if len(missing) == 0:

                status = "🟢 Synchronized"
                color = "#C8E6C9"
                missing_text = "-"

            else:

                status = "🔴 Missing Rounds"
                color = "#FFCDD2"
                missing_text = ", ".join(
                    map(str, missing)
                )

            html += f"""
            <tr style='background:{color}'>
                <td><b>{con.upper()}</b></td>
                <td>{latest}</td>
                <td>{leader}</td>
                <td>{status}</td>
                <td>{missing_text}</td>
            </tr>
            """

        html += "</table>"

        return html

    except Exception as e:

        return f"""
        <h3 style='color:red'>
        Dashboard Error
        </h3>
        <pre>{str(e)}</pre>
        """

# =====================================================
# UI
# =====================================================

dashboard_output = widgets.Output()

refresh_btn = widgets.Button(
    description="Refresh Live",
    button_style="success",
    icon="refresh"
)

def refresh_dashboard(b=None):

    with dashboard_output:

        dashboard_output.clear_output(wait=True)

        display(
            HTML(
                generate_dashboard()
            )
        )

refresh_btn.on_click(refresh_dashboard)

display(
    widgets.HTML(
        "<h3>📊 Election Monitoring Dashboard</h3>"
    )
)

display(refresh_btn)
display(dashboard_output)

refresh_dashboard()

HTML(value='<h3>📊 Election Monitoring Dashboard</h3>')

Button(button_style='success', description='Refresh Live', icon='refresh', style=ButtonStyle())

Output()

In [4]:
!pip install pandas ipywidgets plotly requests

In [5]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

# =====================================================
# GOOGLE SHEET CONFIGURATION
# =====================================================

SHEET_ID = "1-fkXH6w6ehXRa6AXP-qProhjAeTWVj8BmMJHwhZa5Ho"

# CORRECT GIDs
RA_GID = "0"
MASTER_GID = "1836034777"

# =====================================================
# LOAD LIVE GOOGLE SHEET DATA
# =====================================================

def load_data():

    master_url = (
        f"https://docs.google.com/spreadsheets/d/"
        f"{SHEET_ID}/export?format=csv&gid={MASTER_GID}"
    )

    ra_url = (
        f"https://docs.google.com/spreadsheets/d/"
        f"{SHEET_ID}/export?format=csv&gid={RA_GID}"
    )

    master_df = pd.read_csv(master_url)
    ra_df = pd.read_csv(ra_url)

    master_df.columns = (
        master_df.columns.astype(str)
        .str.strip()
    )

    ra_df.columns = (
        ra_df.columns.astype(str)
        .str.strip()
    )

    master_df["Constituency"] = (
        master_df["Constituency"]
        .astype(str)
        .str.strip()
    )

    ra_df["Constituency"] = (
        ra_df["Constituency"]
        .astype(str)
        .str.strip()
    )

    return master_df, ra_df

# =====================================================
# INITIAL LOAD
# =====================================================

master_df, ra_df = load_data()

# =====================================================
# WIDGETS
# =====================================================

dropdown = widgets.Dropdown(
    options=sorted(
        master_df["Constituency"]
        .dropna()
        .unique()
    ),
    description="Select:",
    layout=widgets.Layout(width="350px")
)

refresh_btn = widgets.Button(
    description="🔄 Refresh",
    button_style="success"
)

dashboard = widgets.Output()

# =====================================================
# DASHBOARD FUNCTION
# =====================================================

def update_dashboard(b=None):

    with dashboard:

        clear_output(wait=True)

        try:

            master_df, ra_df = load_data()

            constituency = dropdown.value

            row = master_df[
                master_df["Constituency"]
                .astype(str)
                .str.strip()
                == str(constituency).strip()
            ]

            if row.empty:

                print("No data found.")
                return

            row = row.iloc[0]

            latest_round = row["Latest Round"]
            leader = row["Leading Candidate"]
            missing = row["Missing round"]

            # =================================
            # HEADER
            # =================================

            print("="*90)
            print("🗳️ ELECTION OPERATIONS COMMAND CENTER")
            print("="*90)

            print(f"\n⏰ Last Refresh : {time.strftime('%H:%M:%S')}")
            print(f"📍 Constituency : {constituency}")
            print(f"🔢 Latest Round : {latest_round}")
            print(f"🏆 Leading Candidate : {leader}")
            print(f"⚠️ Missing Rounds : {missing}")

            # =================================
            # RA STATUS
            # =================================

            print("\n")
            print("="*90)
            print("👥 RA STATUS")
            print("="*90)

            if "RA Name" in ra_df.columns:

                for ra in sorted(
                    ra_df["RA Name"]
                    .dropna()
                    .unique()
                ):

                    temp_ra = ra_df[
                        ra_df["RA Name"] == ra
                    ]

                    overtime = 0

                    if "alert status" in temp_ra.columns:

                        overtime = (
                            temp_ra["alert status"]
                            .astype(str)
                            .str.lower()
                            .eq("overtime")
                            .sum()
                        )

                    if overtime > 3:
                        status = "🔴 CRITICAL"

                    elif overtime > 0:
                        status = "🟡 DELAYED"

                    else:
                        status = "🟢 HEALTHY"

                    print(f"{ra} : {status}")

            # =================================
            # HEALTH SCORE
            # =================================

            miss_count = 0

            if pd.notna(missing):

                if str(missing).strip() not in ["", "-", "None"]:

                    miss_count = len(
                        str(missing).split(",")
                    )

            health = max(
                0,
                100 - miss_count * 15
            )

            gauge = go.Figure(
                go.Indicator(
                    mode="gauge+number",
                    value=health,
                    title={
                        "text":
                        "Constituency Health Score"
                    },
                    gauge={
                        "axis":
                        {"range":[0,100]}
                    }
                )
            )

            gauge.show()

            # =================================
            # TOP CANDIDATES
            # =================================

            temp = ra_df[
                ra_df["Constituency"]
                .astype(str)
                .str.strip()
                == str(constituency).strip()
            ]

            if (
                "Candidate Name" in temp.columns
                and "Votes" in temp.columns
            ):

                votes = (
                    temp.groupby(
                        "Candidate Name"
                    )["Votes"]
                    .sum()
                    .reset_index()
                    .sort_values(
                        "Votes",
                        ascending=False
                    )
                    .head(5)
                )

                if len(votes) > 0:

                    bar = px.bar(
                        votes,
                        x="Candidate Name",
                        y="Votes",
                        title="🏆 Top 5 Candidates"
                    )

                    bar.show()

            # =================================
            # PARTY SHARE
            # =================================

            if (
                "Party" in temp.columns
                and "Votes" in temp.columns
            ):

                party = (
                    temp.groupby(
                        "Party"
                    )["Votes"]
                    .sum()
                    .reset_index()
                )

                if len(party) > 0:

                    pie = px.pie(
                        party,
                        names="Party",
                        values="Votes",
                        title="📊 Party Vote Share"
                    )

                    pie.show()

        except Exception as e:

            print("❌ Dashboard Error")
            print(str(e))

# =====================================================
# EVENTS
# =====================================================

dropdown.observe(
    update_dashboard,
    names="value"
)

refresh_btn.on_click(
    update_dashboard
)

# =====================================================
# DISPLAY
# =====================================================

display(
    widgets.VBox([
        widgets.HTML(
            "<h2 style='color:#1565C0'>🗳 Election Operations Command Center</h2>"
        ),
        widgets.HBox([
            dropdown,
            refresh_btn
        ]),
        dashboard
    ])
)

update_dashboard()

In [10]:
!pip install pyttsx3

In [8]:
import pandas as pd
import pyttsx3
from IPython.display import Javascript, display

# ====================================
# GOOGLE SHEET
# ====================================

SHEET_ID = "1-fkXH6w6ehXRa6AXP-qProhjAeTWVj8BmMJHwhZa5Ho"
MASTER_GID = "1836034777"

master_url = (
    f"https://docs.google.com/spreadsheets/d/"
    f"{SHEET_ID}/export?format=csv&gid={MASTER_GID}"
)

master_df = pd.read_csv(master_url)

# ====================================
# VOICE ENGINE
# ====================================

engine = pyttsx3.init()

alerts = []

for _, row in master_df.iterrows():

    constituency = str(row["Constituency"])

    missing = str(row["Missing round"]).strip()

    if (
        missing != ""
        and missing.lower() != "none"
        and missing.lower() != "nan"
    ):

        alerts.append(
            f"{constituency.upper()} : Missing Rounds {missing}"
        )

        speech = (
            f"Attention Team Leader. "
            f"{constituency} needs attention. "
            f"Missing rounds are {missing}"
        )

        engine.say(speech)

# ====================================
# SPEAK ALL ALERTS
# ====================================

if len(alerts) > 0:

    engine.runAndWait()

    popup_text = "\\n\\n".join(alerts)

    display(
        Javascript(
            f"""
            alert(
            '🚨 ELECTION CONTROL ROOM ALERT\\n\\n'
            + '{popup_text}'
            );
            """
        )
    )

else:

    display(
        Javascript(
            """
            alert(
            '✅ ALL CONSTITUENCIES SYNCHRONIZED'
            );
            """
        )
    )

<IPython.core.display.Javascript object>

In [7]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ==========================================================
# CONFIG
# ==========================================================

SHEET_ID = "1-fkXH6w6ehXRa6AXP-qProhjAeTWVj8BmMJHwhZa5Ho"
RA_GID = "0"

TARGET_ROUNDS = 40

# ==========================================================
# LOAD DATA
# ==========================================================

def load_data():

    url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={RA_GID}"

    df = pd.read_csv(url)

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )

    return df

# ==========================================================
# STATUS ENGINE
# ==========================================================

def get_status(rounds):

    pct = rounds / TARGET_ROUNDS * 100

    if pct >= 100:
        return "🏆 COMPLETED"

    elif pct >= 75:
        return "🔵 FINAL STRETCH"

    elif pct >= 50:
        return "🟢 ON TRACK"

    elif pct >= 25:
        return "🟠 CATCHING UP"

    else:
        return "🔴 STARTING"

# ==========================================================
# LOAD INITIAL DATA
# ==========================================================

ra_df = load_data()

constituencies = sorted(
    ra_df["Constituency"]
    .dropna()
    .unique()
)

# ==========================================================
# WIDGETS
# ==========================================================

dropdown = widgets.Dropdown(
    options=constituencies,
    description="Select:",
    layout=widgets.Layout(width="350px")
)

refresh_btn = widgets.Button(
    description="🔄 Refresh",
    button_style="success"
)

dashboard = widgets.Output()

# ==========================================================
# DASHBOARD
# ==========================================================

def update_dashboard(b=None):

    with dashboard:

        clear_output(wait=True)

        ra_df = load_data()

        progress_df = (
            ra_df.groupby("Constituency")["Round"]
            .max()
            .reset_index()
        )

        progress_df.columns = [
            "Constituency",
            "Latest Round"
        ]

        progress_df["Completion"] = (
            progress_df["Latest Round"]
            / TARGET_ROUNDS
            * 100
        )

        progress_df["Remaining"] = (
            TARGET_ROUNDS
            - progress_df["Latest Round"]
        )

        progress_df = progress_df.sort_values(
            "Latest Round",
            ascending=False
        ).reset_index(drop=True)

        selected = dropdown.value

        selected_row = progress_df[
            progress_df["Constituency"] == selected
        ].iloc[0]

        latest_round = int(
            selected_row["Latest Round"]
        )

        completion = round(
            selected_row["Completion"],
            1
        )

        remaining = int(
            selected_row["Remaining"]
        )

        leader = progress_df.iloc[0]["Constituency"]

        laggard = progress_df.iloc[-1]["Constituency"]

        overall_progress = round(
            (
                progress_df["Latest Round"].sum()
                /
                (len(progress_df) * TARGET_ROUNDS)
            ) * 100,
            1
        )

        status = get_status(
            latest_round
        )

        progress_bar = (
            "█" * int(completion / 5)
            +
            "░" * (20 - int(completion / 5))
        )

        # ==================================================
        # HEADER
        # ==================================================

        display(
            HTML(
                f"""
                <div style="
                background:#1565C0;
                color:white;
                padding:20px;
                border-radius:12px;
                margin-bottom:15px;
                ">
                <h1>
                🗳 Election Mission Control
                </h1>
                <h3>
                Kantar Election Operations Intelligence Platform
                </h3>
                </div>
                """
            )
        )

        # ==================================================
        # KPI CARDS
        # ==================================================

        display(
            HTML(
                f"""
                <div style="
                display:flex;
                gap:15px;
                margin-bottom:20px;
                ">

                <div style="
                background:#1565C0;
                color:white;
                padding:15px;
                border-radius:10px;
                width:250px;
                ">
                <h3>📈 Overall Progress</h3>
                <h1>{overall_progress}%</h1>
                </div>

                <div style="
                background:#2E7D32;
                color:white;
                padding:15px;
                border-radius:10px;
                width:250px;
                ">
                <h3>🏆 Leader</h3>
                <h2>{leader}</h2>
                </div>

                <div style="
                background:#D32F2F;
                color:white;
                padding:15px;
                border-radius:10px;
                width:250px;
                ">
                <h3>🚨 Needs Attention</h3>
                <h2>{laggard}</h2>
                </div>

                </div>
                """
            )
        )

        # ==================================================
        # SELECTED CONSTITUENCY CARD
        # ==================================================

        display(
            HTML(
                f"""
                <div style="
                background:#F5F7FA;
                padding:25px;
                border-radius:15px;
                border-left:8px solid #1565C0;
                margin-bottom:20px;
                ">

                <h2>🚀 {selected.upper()}</h2>

                <h3>{status}</h3>

                <h2>
                {latest_round}/{TARGET_ROUNDS}
                Rounds Completed
                </h2>

                <div style="
                font-size:22px;
                font-family:monospace;
                color:#1565C0;
                ">
                {progress_bar}
                </div>

                <br>

                <b>Completion:</b>
                {completion}%

                <br><br>

                <b>Remaining:</b>
                {remaining} Rounds

                <br><br>

                <b>Next Target:</b>
                Round {latest_round + 1}

                </div>
                """
            )
        )

        # ==================================================
        # LIVE RANKING TABLE
        # ==================================================

        ranking_html = """
        <h2>🏁 Live Constituency Ranking</h2>

        <table
        style="
        width:100%;
        border-collapse:collapse;
        text-align:left;
        ">
        """

        medals = [
            "🥇",
            "🥈",
            "🥉",
            "4️⃣",
            "5️⃣",
            "6️⃣"
        ]

        for i, row in progress_df.iterrows():

            rank = medals[i]

            progress = (
                "█" * int(row["Completion"] / 5)
                +
                "░" * (
                    20 -
                    int(row["Completion"] / 5)
                )
            )

            ranking_html += f"""
            <tr>
            <td style="padding:8px;">
            {rank}
            </td>

            <td style="padding:8px;">
            <b>{row['Constituency']}</b>
            </td>

            <td style="padding:8px;">
            {progress}
            </td>

            <td style="padding:8px;">
            {int(row['Latest Round'])}/40
            </td>

            </tr>
            """

        ranking_html += "</table>"

        display(
            HTML(
                ranking_html
            )
        )

        # ==================================================
        # AI BRIEFING
        # ==================================================

        display(
            HTML(
                f"""
                <br>

                <div style="
                background:#FFF8E1;
                padding:20px;
                border-radius:12px;
                border-left:8px solid #FFA000;
                ">

                <h2>
                🤖 Election GPT Briefing
                </h2>

                <p>
                🏆 <b>{leader}</b>
                currently leads the election race.
                </p>

                <p>
                🚨 <b>{laggard}</b>
                requires immediate attention.
                </p>

                <p>
                🎯 <b>{selected}</b>
                has reached Round
                <b>{latest_round}</b>.
                </p>

                <p>
                ⏳ Only
                <b>{remaining}</b>
                rounds remain to reach the target.
                </p>

                <p>
                📈 Current completion:
                <b>{completion}%</b>
                </p>

                <p>
                💡 Recommendation:
                Focus on reaching
                <b>Round {latest_round + 1}</b>
                immediately.
                </p>

                </div>
                """
            )
        )

# ==========================================================
# EVENTS
# ==========================================================

dropdown.observe(
    update_dashboard,
    names="value"
)

refresh_btn.on_click(
    update_dashboard
)

# ==========================================================
# DISPLAY
# ==========================================================

display(
    widgets.VBox([
        widgets.HBox([
            dropdown,
            refresh_btn
        ]),
        dashboard
    ])
)

update_dashboard()